In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

df = pd.read_excel('Online Retail.xlsx')
df.columns = df.columns.str.strip()

# chuan hoa va loai bo hoa don tinh dung (ky tu 'C')
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df = df[~df['InvoiceNo'].str.contains('C', na=False)]


# Loc du lieu Germany, gom nhom theo hoa don + ten mat hang
basket = (df[df['Country'] == "Germany"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0).set_index('InvoiceNo'))
print("Bang gio hang Germany:", basket.shape, "| So hoa don:", basket.shape[0])

# Chuyen doi sang hot encoding (0/1) de dua vao thuat toan Apriori
hot_encoded = basket.map(lambda x: 1 if x > 0 else 0)

# Tim tap pho bien bang thuat toan Apriori voi min_sup = 0.05
frequent_itemsets = apriori(hot_encoded, min_support=0.05, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index()
print("\nSo tap pho bien tim duoc voi min_sup = 0.05:", len(frequent_itemsets))
display(frequent_itemsets.head(20))

# Tao luat ket hop voi min_conf = 0.50
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.50)
rules = rules.sort_values(['lift', 'confidence'], ascending=False).reset_index(drop=True)
print("\nSo luat ket hop thoa man min_conf = 0.50:", len(rules))
display(rules.head(20)[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

print("\n--- Top 10 luat ket hop theo confidence ---")
display(rules.head(10)[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
print("\n--- Top 10 luat ket hop theo lift ---")
display(rules.head(10)[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
